In [1]:
# -- 0. SETUP --
import os
from pyspark.sql import functions as F
from pyspark.sql.types import (StringType, IntegerType, FloatType, ArrayType,
                                StructType, StructField, LongType, DoubleType)

from config import build_spark_session
spark = build_spark_session("MyDigitalTwin-BehavioralClustering")
spark.sparkContext.setLogLevel("WARN")

import sys as _sys, os as _os
_d = _os.path.abspath('')
while not _os.path.exists(_os.path.join(_d, 'config.py')):
    _p = _os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
if _d not in _sys.path: _sys.path.insert(0, _d)
from config import WAREHOUSE

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(name):
    return spark.read.format("delta").load(os.path.join(WAREHOUSE, name))


Warehouse: /opt/spark/data/warehouse


26/04/27 22:54:46 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


---
## PARTIE B — Behavioral Clustering

On rassemble toutes les activités avec leurs features temporelles : heure, jour de la semaine, plateforme, poids d'interaction.

**Features retenues (V1 finale)** : `hour_sin/cos`, `weekday_sin/cos`, `weight`, `platform_ohe`

> **Note V2 testée** : retrait de la plateforme pour des profils purement temporels → 6 clusters quasi-identiques (doublons "Soir Semaine" × 2, "Après-midi Weekend" × 2, Silhouette 0.33). La plateforme est un signal comportemental réel — mode Spotify journée ≠ soirée multi-plateforme. V1 conservée.

In [2]:
# ── B1. CHARGEMENT DES FEATURES COMPORTEMENTALES ──────────────────────────────
# Features retenues : hour (cyclique), weekday (cyclique), platform (OHE), weight
# Sources avec colonnes event_hour + event_weekday disponibles
#
# ⚠ Après ajout de nouvelles sources, relancer B3-B5 et mettre à jour BEH_LABELS
#   en lisant les résultats de B4 (plateforme dominante + heure par cluster).

def safe_read(name, hour_col="event_hour", weekday_col="event_weekday",
              platform_name=None, weight_val=None, limit=None):
    """Charge une table warehouse et retourne (hour, weekday, platform, weight)."""
    try:
        df = read_table(name)
        w_col = F.col("interaction_weight").cast(FloatType()) if "interaction_weight" in df.columns \
                else F.lit(weight_val or 1.0).cast(FloatType())
        sel = df.select(
            F.col(hour_col).alias("hour"),
            F.col(weekday_col).alias("weekday"),
            F.lit(platform_name or name).alias("platform"),
            w_col.alias("weight"),
        )
        if limit:
            sel = sel.limit(limit)
        return sel
    except Exception as e:
        print(f"⚠ {name} ignoré : {e}")
        return None

sources = [
    # ── Google / YouTube ──────────────────────────────────────────────────────
    safe_read("youtube_watch",    platform_name="youtube"),
    safe_read("google_searches",  platform_name="google",   weight_val=1.0),
    safe_read("google_chrome",    platform_name="chrome",   weight_val=1.0),
    # ── Spotify ───────────────────────────────────────────────────────────────
    safe_read("spotify_streams",  hour_col="listen_hour", weekday_col="listen_weekday",
              platform_name="spotify"),
    # ── Netflix (pas d'heure → 21h par défaut) ───────────────────────────────
    read_table("netflix_views").select(
        F.lit(21).cast(IntegerType()).alias("hour"),
        F.col("watch_weekday").alias("weekday"),
        F.lit("netflix").alias("platform"),
        F.col("interaction_weight").cast(FloatType()).alias("weight"),
    ),
    # ── TikTok — 3 sources (limites pour éviter de noyer) ────────────────────
    safe_read("tiktok_watch",     platform_name="tiktok",        limit=2000),
    safe_read("tiktok_likes",     platform_name="tiktok_likes",  limit=2000),
    safe_read("tiktok_searches",  platform_name="tiktok_search", weight_val=1.0, limit=1000),
    # ── Instagram — 4 sources (limites) ──────────────────────────────────────
    safe_read("instagram_likes",    platform_name="instagram",        limit=2000),
    safe_read("instagram_saved",    platform_name="instagram_saved",  limit=500),
    safe_read("instagram_comments", platform_name="instagram_comment",
              weight_val=2.5, limit=500),
    # Nouvelles tables (disponibles après re-ingestion instagram.ipynb)
    safe_read("instagram_posts_viewed",   platform_name="ig_posts",   limit=2000),
    safe_read("instagram_videos_watched", platform_name="ig_videos",  limit=2000),
    safe_read("instagram_story_likes",    platform_name="ig_stories",
              weight_val=1.5, limit=1000),
    safe_read("instagram_searches",       platform_name="ig_search",
              weight_val=1.0, limit=500),
    # ── Twitter ───────────────────────────────────────────────────────────────
    safe_read("twitter_tweets",   platform_name="twitter"),
]

valid_sources = [s for s in sources if s is not None]
behavioral_raw = valid_sources[0]
for s in valid_sources[1:]:
    behavioral_raw = behavioral_raw.union(s)

behavioral_raw = behavioral_raw.filter(
    F.col("hour").isNotNull() & F.col("weekday").isNotNull()
)

print(f"Total events comportementaux : {behavioral_raw.count():,}")
behavioral_raw.groupBy("platform").count().orderBy(F.desc("count")).show(20)


⚠ instagram_searches ignoré : [PATH_NOT_FOUND] Path does not exist: /opt/spark/data/warehouse/instagram_searches.


26/04/27 22:55:09 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total events comportementaux : 206,632


+-----------------+------+
|         platform| count|
+-----------------+------+
|          spotify|123033|
|           google| 55827|
|          youtube| 14076|
|          netflix|  4288|
|           tiktok|  2000|
|     tiktok_likes|  2000|
|        instagram|  2000|
|    tiktok_search|  1000|
|        ig_videos|   625|
|         ig_posts|   609|
|       ig_stories|   417|
|           chrome|   338|
|          twitter|   321|
|instagram_comment|    81|
|  instagram_saved|    17|
+-----------------+------+



In [3]:
# -- B2. PROFILS TEMPORELS PAR PLATEFORME --
# Chaque plateforme devient un vecteur de 5 features :
# (% matin, % apres-midi, % soir, % nuit, % weekend)
# K-Means sur 16 profils au lieu de 226k events bruts.

from pyspark.ml.feature import VectorAssembler, StandardScaler

profiled = behavioral_raw \
    .withColumn("slot",
        F.when((F.col("hour") >= 5)  & (F.col("hour") < 12), "morning")
         .when((F.col("hour") >= 12) & (F.col("hour") < 18), "afternoon")
         .when((F.col("hour") >= 18) & (F.col("hour") < 23), "evening")
         .otherwise("night")
    ) \
    .withColumn("is_weekend", F.when(F.col("weekday") >= 6, 1.0).otherwise(0.0))

platform_total = profiled.groupBy("platform").agg(
    F.count("*").alias("total"),
    F.round(F.avg("hour"), 2).alias("avg_hour"),
    F.round(F.avg("weekday"), 2).alias("avg_weekday"),
    (F.sum("is_weekend") / F.count("*")).alias("weekend_pct"),
)

slot_counts = profiled.groupBy("platform", "slot").agg(F.count("*").alias("cnt"))
slot_pct = slot_counts.join(platform_total.select("platform", "total"), "platform") \
    .withColumn("pct", F.col("cnt") / F.col("total")) \
    .groupBy("platform") \
    .pivot("slot", ["morning", "afternoon", "evening", "night"]) \
    .agg(F.first("pct")) \
    .fillna(0.0)

platform_profiles = slot_pct.join(platform_total, "platform")

assembler = VectorAssembler(
    inputCols=["morning", "afternoon", "evening", "night", "weekend_pct"],
    outputCol="raw_features"
)
platform_profiles = assembler.transform(platform_profiles)

scaler = StandardScaler(inputCol="raw_features", outputCol="features", withMean=True, withStd=True)
scaler_model = scaler.fit(platform_profiles)
platform_profiles = scaler_model.transform(platform_profiles)

print(f"Profils de plateformes : {platform_profiles.count()} plateformes")
platform_profiles.select("platform", "morning", "afternoon", "evening", "night", "weekend_pct", "avg_hour").show(20, truncate=False)

Profils de plateformes : 15 plateformes


+-----------------+---------------------+-------------------+-------------------+--------------------+--------------------+--------+
|platform         |morning              |afternoon          |evening            |night               |weekend_pct         |avg_hour|
+-----------------+---------------------+-------------------+-------------------+--------------------+--------------------+--------+
|spotify          |0.2619297261710273   |0.3253923744036153 |0.2593125421634846 |0.15336535726187284 |0.29589622296457047 |13.18   |
|tiktok_likes     |0.1805               |0.2105             |0.3565             |0.2525              |0.3105              |13.69   |
|ig_videos        |0.4656               |0.3232             |0.1968             |0.0144              |0.2864              |13.21   |
|ig_stories       |0.20863309352517986  |0.31894484412470026|0.381294964028777  |0.09112709832134293 |0.2997601918465228  |15.09   |
|instagram_saved  |0.11764705882352941  |0.7058823529411765 |0.117647

In [4]:
# -- B3. KMEANS SUR PROFILS DE PLATEFORMES --
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from config import K_BEHAVIORAL

kmeans = KMeans(featuresCol="features", predictionCol="beh_cluster",
                k=K_BEHAVIORAL, seed=42, maxIter=100)

print(f"Training K-Means sur profils (k={K_BEHAVIORAL})...")
km_model = kmeans.fit(platform_profiles)
beh_df = km_model.transform(platform_profiles)

evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="beh_cluster")
sil = evaluator.evaluate(beh_df)
print(f"Silhouette Score (profils): {sil:.4f}")

(beh_df.select("platform", "beh_cluster", "avg_hour", "weekend_pct",
              "morning", "afternoon", "evening", "night")
        .orderBy("beh_cluster", "platform").show(20, truncate=False))

Training K-Means sur profils (k=4)...


26/04/27 22:58:23 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Silhouette Score (profils): 0.5470


+-----------------+-----------+--------+--------------------+---------------------+-------------------+-------------------+--------------------+
|platform         |beh_cluster|avg_hour|weekend_pct         |morning              |afternoon          |evening            |night               |
+-----------------+-----------+--------+--------------------+---------------------+-------------------+-------------------+--------------------+
|google           |0          |15.25   |0.28097515539076073 |0.13317928600856216  |0.336199330073262  |0.3674566070181095 |0.16316477690006628 |
|ig_posts         |0          |14.21   |0.24466338259441708 |0.24958949096880131  |0.5172413793103449 |0.2134646962233169 |0.019704433497536946|
|ig_stories       |0          |15.09   |0.2997601918465228  |0.20863309352517986  |0.31894484412470026|0.381294964028777  |0.09112709832134293 |
|ig_videos        |0          |13.21   |0.2864              |0.4656               |0.3232             |0.1968             |0.0144 

In [5]:
# -- B4. CARACTERISATION DES CLUSTERS --

beh_cluster_info = []
cluster_data = beh_df.groupBy("beh_cluster").agg(
    F.collect_list("platform").alias("platforms"),
    F.round(F.avg("avg_hour"), 1).alias("avg_hour"),
    F.round(F.avg("avg_weekday"), 1).alias("avg_weekday"),
    F.round(F.avg("weekend_pct"), 3).alias("weekend_pct"),
    F.sum("total").alias("item_count"),
).orderBy("beh_cluster").collect()

for row in cluster_data:
    cid    = row["beh_cluster"]
    avg_h  = row["avg_hour"] or 0.0
    avg_wd = row["avg_weekday"] or 0.0
    count  = row["item_count"]
    plats  = row["platforms"]

    h = round(avg_h)
    if 5 <= h < 12:    period = "Matin"
    elif 12 <= h < 18: period = "Apres-midi"
    elif 18 <= h < 23: period = "Soir"
    else:              period = "Nuit"

    day_type = "Weekend" if row["weekend_pct"] > 0.35 else "Semaine"

    beh_cluster_info.append({
        "cluster_id":    cid,
        "item_count":    count,
        "avg_hour":      float(avg_h),
        "avg_weekday":   float(avg_wd),
        "time_period":   period,
        "day_type":      day_type,
        "top_platforms": plats,
    })

    print(f"[Cluster {cid}] {count:,} events | {period} . {day_type} | {plats}")

[Cluster 0] 201,989 events | Apres-midi . Semaine | ['youtube', 'google', 'spotify', 'tiktok', 'tiktok_likes', 'tiktok_search', 'instagram', 'instagram_comment', 'ig_posts', 'ig_videos', 'ig_stories', 'twitter']
[Cluster 1] 338 events | Apres-midi . Semaine | ['chrome']
[Cluster 2] 17 events | Apres-midi . Semaine | ['instagram_saved']
[Cluster 3] 4,288 events | Soir . Semaine | ['netflix']


In [6]:
# -- B5. LABELS DES CLUSTERS --
# Placeholders — a ajuster apres lecture des resultats B4.

BEH_LABELS = {
    0: {"label": "📱 Réseaux & Médias · Après-midi",  "emoji": "📱"},
    1: {"label": "🌐 Navigation Chrome · Après-midi", "emoji": "🌐"},
    2: {"label": "👻 Sauvegardes IG · Rare",          "emoji": "👻"},
    3: {"label": "🎬 Netflix · Soirée",               "emoji": "🎬"},
}

for info in beh_cluster_info:
    cid = info["cluster_id"]
    print(f"  Cluster {cid}: {info['time_period']} . {info['day_type']} | {info['top_platforms']}")

  Cluster 0: Apres-midi . Semaine | ['youtube', 'google', 'spotify', 'tiktok', 'tiktok_likes', 'tiktok_search', 'instagram', 'instagram_comment', 'ig_posts', 'ig_videos', 'ig_stories', 'twitter']
  Cluster 1: Apres-midi . Semaine | ['chrome']
  Cluster 2: Apres-midi . Semaine | ['instagram_saved']
  Cluster 3: Soir . Semaine | ['netflix']


### Écriture warehouse — behavioral_clusters

**Merge key**: `cluster_id` (identifiant unique par cluster comportemental)  
**Stratégie**: Delta MERGE INTO — crée la table au 1er run, met à jour les clusters existants et insère les nouveaux aux runs suivants.  
**Idempotent**: oui — relancer le notebook avec les mêmes données ne crée pas de doublons.

In [7]:
# -- B6. ECRITURE behavioral_clusters (Delta MERGE) --
from delta.tables import DeltaTable

beh_rows = []
for info in beh_cluster_info:
    cid = info["cluster_id"]
    beh_rows.append((
        cid,
        BEH_LABELS.get(cid, {}).get("label", f"Profil {cid}"),
        BEH_LABELS.get(cid, {}).get("emoji", "?"),
        float(info["avg_hour"]),
        float(info["avg_weekday"]),
        info["time_period"],
        info["day_type"],
        info["top_platforms"],
        info["item_count"]
    ))

schema_beh = StructType([
    StructField("cluster_id",    IntegerType(), False),
    StructField("label",         StringType(),  False),
    StructField("emoji",         StringType(),  True),
    StructField("avg_hour",      DoubleType(),  True),
    StructField("avg_weekday",   DoubleType(),  True),
    StructField("time_period",   StringType(),  True),
    StructField("day_type",      StringType(),  True),
    StructField("top_platforms", ArrayType(StringType()), True),
    StructField("item_count",    LongType(),    True),
])

beh_clusters_df = spark.createDataFrame(beh_rows, schema_beh)
out_path = os.path.join(WAREHOUSE, "behavioral_clusters")

# MERGE INTO si la table existe, sinon creation initiale
if DeltaTable.isDeltaTable(spark, out_path):
    delta_tbl = DeltaTable.forPath(spark, out_path)
    (
        delta_tbl.alias("target")
        .merge(beh_clusters_df.alias("source"), "target.cluster_id = source.cluster_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    beh_clusters_df.write.format("delta").mode("overwrite").save(out_path)

print(f"Ecrit dans : {out_path}")
beh_clusters_df.show(truncate=50)

Ecrit dans : /opt/spark/data/warehouse/behavioral_clusters
+----------+---------------------------------+-----+--------+-----------+-----------+--------+--------------------------------------------------+----------+
|cluster_id|                            label|emoji|avg_hour|avg_weekday|time_period|day_type|                                     top_platforms|item_count|
+----------+---------------------------------+-----+--------+-----------+-----------+--------+--------------------------------------------------+----------+
|         0| 📱 Réseaux & Médias · Après-midi|   📱|    14.5|        4.0| Apres-midi| Semaine|[youtube, google, spotify, tiktok, tiktok_likes...|    201989|
|         1|🌐 Navigation Chrome · Après-midi|   🌐|    13.9|        3.5| Apres-midi| Semaine|                                          [chrome]|       338|
|         2|         👻 Sauvegardes IG · Rare|   👻|    15.7|        3.9| Apres-midi| Semaine|                                 [instagram_saved]|        17|
|    

In [8]:
spark.stop()
print("Spark session fermée. Notebook terminé.")

Spark session fermée. Notebook terminé.
